#### IMPORT

In [16]:
# ========== SETUP FOR GOOGLE COLAB ==========
import os
import sys

colab_mode = False

# Mount Google Drive (required for Colab)
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        colab_mode = True
        print("✓ Google Drive mounted successfully")
    except ValueError as e:
        print(f"⚠ Could not mount Google Drive: {e}")
        print("  (This is normal if you're testing locally)")
except ImportError:
    print("⚠ Not running in Google Colab (local development mode)")

# Set working directory
if colab_mode:
    # On Colab, files are in /content/drive/MyDrive/
    workspace_path = '/content/drive/MyDrive/Crawler'
else:
    # Local fallback
    workspace_path = os.getcwd()

if os.path.exists(workspace_path):
    os.chdir(workspace_path)
    print(f"✓ Changed directory to: {os.getcwd()}")
else:
    print(f"⚠ Workspace path not found: {workspace_path}")
    print(f"  Using current directory: {os.getcwd()}")

# Add to Python path
sys.path.insert(0, os.getcwd())

# ========== IMPORT LIBRARIES ==========
import pandas as pd
from transformers import pipeline

# Try to import project modules
try:
    from Libraries import Common_MyUtils as MyUtils
    from Libraries import URL_Crawler as Crawler
    from Libraries import URL_Sorter as Sorter
    from Libraries import URL_Processor as Processor
    print("✓ Successfully imported from Libraries")
except ImportError as e:
    print(f"⚠ Could not import Libraries: {e}")
    print("Note: For Colab, make sure Libraries folder is in your Google Drive")
    print("   or upload it manually")
    
print("\n✓ Initialization completed")

⚠ Could not mount Google Drive: mount failed
  (This is normal if you're testing locally)
✓ Changed directory to: /content
⚠ Could not import Libraries: No module named 'Libraries'
Note: For Colab, make sure Libraries folder is in your Google Drive
   or upload it manually

✓ Initialization completed


#### CRAWL CONFIG

In [17]:
# ========== CONFIGURATION ==========
import json

# Handle missing categories.json file
RESOURCE_FILE = "Resource/categories.json"

# Create sample categories if file doesn't exist
if not os.path.exists(RESOURCE_FILE):
    print(f"⚠ Creating sample {RESOURCE_FILE} (missing original)")
    os.makedirs("Resource", exist_ok=True)
    sample_categories = {
        "thoi-su": {"name": "Thời sự"},
        "the-gioi": {"name": "Thế giới"},
        "kinh-doanh": {"name": "Kinh doanh"},
        "bat-dong-san": {"name": "Bất động sản"},
        "khoa-hoc": {"name": "Khoa học"},
        "giai-tri": {"name": "Giải trí"},
        "the-thao": {"name": "Thể thao"},
        "phap-luat": {"name": "Pháp luật"},
        "giao-duc": {"name": "Giáo dục"},
        "suc-khoe": {"name": "Sức khỏe"},
        "doi-song": {"name": "Đời sống"},
        "oto-xe-may": {"name": "Ôtô - Xe máy"}
    }
    with open(RESOURCE_FILE, 'w', encoding='utf-8') as f:
        json.dump(sample_categories, f, ensure_ascii=False, indent=2)
    print(f"✓ Created {RESOURCE_FILE}")

# Read categories
try:
    with open(RESOURCE_FILE, 'r', encoding='utf-8') as f:
        type_dict = json.load(f)
except:
    type_dict = {}
    print("⚠ Could not load categories")

# Configuration
my_config = {
    "BASE_URL": "https://vnexpress.net",
    "MIN_YEAR": 2020,
    "MIN_WORDS": 200,
    "MAX_WORDS": 1000,
    "TARGET_ARTICLES_PER_SUBTYPE": 5,
    "MAX_CONCURRENT_WORKERS": 6,
    "VALIDATION_ARTICLES_COUNT": 5,
    "PROGRESS_TIMEOUT": 10,
    "ARTICLE_TIMEOUT": 10,
    "MAX_CONSECUTIVE_FAILURES": 3,
    "URL_MAX_SUBCATEGORY_FAILURES": 3, 
    "TYPE_DICT": type_dict
}

# Paths
DATABASE = "Database/Summary"
pageName = "VNExpress"
URL_DIR = "Resource"
CATE_FILE = f"{URL_DIR}/{pageName}_CATE.json"
DICT_FILE = f"{URL_DIR}/{pageName}_DICT.json"
URLS_PATH = f"{URL_DIR}/{pageName}_URLS"
JSON_PATH = f"{DATABASE}/JSON/{pageName}"
XLSX_PATH = f"{DATABASE}/XLSX/{pageName}"

# Create directories
os.makedirs(URL_DIR, exist_ok=True)
os.makedirs(os.path.join(DATABASE, "JSON"), exist_ok=True)
os.makedirs(os.path.join(DATABASE, "XLSX"), exist_ok=True)

# Load validated dictionary
try:
    with open(DICT_FILE, 'r', encoding='utf-8') as f:
        validated_dict = json.load(f)
except FileNotFoundError:
    validated_dict = {}
    print(f"⚠ {DICT_FILE} not found yet")

# Categories to crawl
categories_to_crawl = [
    'thoi-su', 
    'the-gioi', 
    'kinh-doanh', 
    'bat-dong-san', 
    'khoa-hoc', 
    'giai-tri', 
    'the-thao', 
    'phap-luat', 
    'giao-duc', 
    'suc-khoe', 
    'doi-song', 
    'oto-xe-may'
]

print("✓ Configuration loaded successfully")

⚠ Creating sample Resource/categories.json (missing original)
✓ Created Resource/categories.json
⚠ Resource/VNExpress_DICT.json not found yet
✓ Configuration loaded successfully


#### CRAWL FUNCTIONS

In [18]:
# --- Giai đoạn 1: Lấy danh sách chuyên mục hợp lệ ---
def getCategories():
    try:
        validator = Crawler.CategoryValidator(config=my_config)
        valid_categories = validator.run()
        MyUtils.write_json(valid_categories, DICT_FILE)
        print("✓ Categories validation completed")
    except AttributeError:
        print("⚠ getCategories: Crawler module not available")
    except Exception as e:
        print(f"✗ Error in getCategories: {e}")

In [20]:
# --- Giai đoạn 2: Thu thập URL ---
def getDict():
    try:
        for category_name in categories_to_crawl:

            URLS_FILE = f"{URLS_PATH}_{category_name}.json"
            JSON_FILE = f"{JSON_PATH}_{category_name}.jsonl"

            print("\n" + "="*50)
            print(f"BẮT ĐẦU LẤY URL CHO CATEGORY: {category_name.upper()}")
            print("="*50)

            # 1. Lấy danh sách TẤT CẢ các URL đã tồn tại từ cả 2 nguồn
            urls_in_url_file = Processor.get_urls_from_url_file(URLS_FILE)
            urls_in_jsonl_file = Processor.get_existing_article_urls(JSON_FILE)
            all_existing_urls = urls_in_url_file.union(urls_in_jsonl_file)
            
            print(f"Đã tìm thấy {len(all_existing_urls)} URL đã tồn tại cho category này.")

            # 2. Chạy collector để thu thập tất cả URL có thể có
            url_collector = Crawler.UrlCollector(config=my_config)
            all_possible_urls = url_collector.run(
                valid_subcategories=validated_dict,
                categories_to_process=[category_name]
            )

            # 3. Lọc ra chỉ những URL thực sự mới
            new_urls_info = [
                info for info in all_possible_urls 
                if info['url'] not in all_existing_urls
            ]
            
            print(f"Thu thập được {len(new_urls_info)} URL mới.")

            # 4. Ghi lại file URLS với dữ liệu cũ + mới
            if new_urls_info:
                existing_urls_info = MyUtils.read_json(URLS_FILE)
                combined_urls_info = existing_urls_info + new_urls_info
                MyUtils.write_json(combined_urls_info, URLS_FILE)
    except AttributeError:
        print("⚠ getDict: Processor/Crawler module not available")
    except Exception as e:
        print(f"✗ Error in getDict: {e}")
    
# === END ===

In [21]:
# --- Giai đoạn 3: Crawl nội dung bài viết ---
def runCrawl():
    try:
        for category_name in categories_to_crawl:

            URLS_FILE = f"{URLS_PATH}_{category_name}.json"
            JSON_FILE = f"{JSON_PATH}_{category_name}.jsonl"

            print("\n" + "="*50)
            print(f"BẮT ĐẦU CRAWL CHO CATEGORY: {category_name.upper()}")
            print("="*50)

            urls_for_category = MyUtils.read_json(URLS_FILE)
            if not urls_for_category:
                print("Không có URL nào trong file để crawl. Bỏ qua.")
                continue

            # 1. Xác định các URL đã được xử lý từ trước
            existing_urls_in_jsonl = Processor.get_existing_article_urls(JSON_FILE)
            print(f"Tìm thấy {len(existing_urls_in_jsonl)} bài viết đã tồn tại trong file {JSON_FILE}.")
            
            # 2. Chạy crawler
            article_crawler = Crawler.ArticleCrawler(config=my_config)
            new_articles, successfully_crawled_urls = article_crawler.run(
                urls_to_crawl=urls_for_category,
                category=category_name,
                existing_article_urls=existing_urls_in_jsonl
            )
            
            # 3. Lưu dữ liệu mới nếu có
            if new_articles:
                MyUtils.insert_json(new_articles, JSON_FILE)
                print(f"Đã lưu {len(new_articles)} bài báo mới vào {JSON_FILE}.")

            # 4. Xóa các URL đã được xử lý khỏi file URLS_FILE
            processed_urls = existing_urls_in_jsonl.union(set(successfully_crawled_urls))
            if processed_urls:
                # Lọc ra danh sách các URL còn lại
                remaining_urls = [
                    info for info in urls_for_category 
                    if info['url'] not in processed_urls
                ]
                
                # Ghi đè lại file URLS_FILE
                MyUtils.write_json(remaining_urls, URLS_FILE)
                print(f"Đã xóa {len(processed_urls)} URL đã xử lý. Còn lại {len(remaining_urls)} URL trong hàng đợi.")
    except AttributeError:
        print("⚠ runCrawl: Crawler/Processor module not available")
    except Exception as e:
        print(f"✗ Error in runCrawl: {e}")

# === END ===

#### RESIGN DATA FUNCTIONS

In [22]:
# --- Giai đoạn 4: Sort bài viết ---
def sortCrawl():
    try:
        for category_name in categories_to_crawl:

            URLS_FILE = f"{URLS_PATH}_{category_name}.json"
            JSON_FILE = f"{JSON_PATH}_{category_name}.jsonl"

            print("\n" + "="*50)
            print(f"BẮT ĐẦU SORT CHO CATEGORY: {category_name.upper()}")
            print("="*50)

            # --- Sắp xếp URLS_FILE ---
            urls_data = MyUtils.read_json(URLS_FILE)
            if urls_data:
                sorted_urls = Processor.heapSort(urls_data, key_func=Processor.get_url_key)
                MyUtils.write_json(sorted_urls, URLS_FILE)

            # --- Sắp xếp JSON_FILE ---
            all_articles = MyUtils.read_jsonl(JSON_FILE)
            if all_articles:
                sorter = Sorter.ArticleSorter(categories_file_path=CATE_FILE)
                sorted_articles = sorter.sort_and_deduplicate(all_articles)
                MyUtils.write_jsonl(sorted_articles, JSON_FILE)
            
            print(f"-> Đã hoàn tất sắp xếp cho category: {category_name}")
    except AttributeError:
        print("⚠ sortCrawl: Sorter/Processor module not available")
    except Exception as e:
        print(f"✗ Error in sortCrawl: {e}")

# === END ===

In [23]:
# --- Giai đoạn 5: Tổng hợp và Hoàn thiện Dữ liệu (Logic cuối cùng, tự động) ---
def finalizeData():
    try:
        # Định nghĩa đường dẫn cho các file master
        MASTER_JSONL_FILE = f"{JSON_PATH}.jsonl"
        MASTER_JSON_FILE = f"{JSON_PATH}.json" 
        MASTER_XLSX_FILE = f"{XLSX_PATH}.xlsx"

        categories_dict = MyUtils.read_json(CATE_FILE)
        if not categories_dict:
            return
        
        all_categories = list(categories_dict.keys())

        # Xóa các file tổng hợp cũ để bắt đầu lại từ đầu
        for f in [MASTER_JSONL_FILE, MASTER_JSON_FILE]:
            if os.path.exists(f):
                os.remove(f)

        # 1. Gộp dữ liệu từ các file JSONL của tất cả category tìm được
        print("\n--- Bước 1: Gộp dữ liệu từ các file category ---")
        for category_name in all_categories: # Sử dụng danh sách vừa đọc được
            JSON_FILE = f"{JSON_PATH}_{category_name}.jsonl"
            # Chỉ xử lý nếu file của category đó tồn tại
            if os.path.exists(JSON_FILE):
                articles_data = MyUtils.read_jsonl(JSON_FILE)
                if articles_data:
                    MyUtils.insert_json(articles_data, MASTER_JSONL_FILE)

        # 2. Sắp xếp và xóa trùng lặp file tổng hợp
        print("\n--- Bước 2: Sắp xếp và dọn dẹp file tổng hợp ---")
        all_merged_articles = MyUtils.read_jsonl(MASTER_JSONL_FILE)
        if all_merged_articles:
            sorter = Sorter.ArticleSorter(categories_file_path=CATE_FILE)
            final_sorted_articles = sorter.sort_and_deduplicate(all_merged_articles)

            # 3. Ghi kết quả ra cả 2 định dạng file
            print("\n--- Bước 3: Ghi file tổng hợp ở cả 2 định dạng (.jsonl và .json) ---")
            MyUtils.write_jsonl(final_sorted_articles, MASTER_JSONL_FILE)
            MyUtils.write_json(final_sorted_articles, MASTER_JSON_FILE)

            # 4. Xuất file Excel cuối cùng
            print("\n--- Bước 4: Xuất file Excel tổng hợp ---")
            MyUtils.convert_to_xlsx(MASTER_JSON_FILE, MASTER_XLSX_FILE)
    except AttributeError:
        print("⚠ finalizeData: Sorter module not available")
    except Exception as e:
        print(f"✗ Error in finalizeData: {e}")

# === END ===

#### RUN

In [24]:
print("="*50)
print("RUNNING PIPELINE...")
print("="*50)

# Run all stages
getCategories()
getDict()
runCrawl()
sortCrawl()
finalizeData()

print("\n" + "="*50)
print("✓ Pipeline completed")
print("="*50)

RUNNING PIPELINE...
✗ Error in getCategories: name 'Crawler' is not defined

BẮT ĐẦU LẤY URL CHO CATEGORY: THOI-SU
✗ Error in getDict: name 'Processor' is not defined

BẮT ĐẦU CRAWL CHO CATEGORY: THOI-SU
✗ Error in runCrawl: [Errno 2] No such file or directory: 'Resource/VNExpress_URLS_thoi-su.json'

BẮT ĐẦU SORT CHO CATEGORY: THOI-SU
✗ Error in sortCrawl: [Errno 2] No such file or directory: 'Resource/VNExpress_URLS_thoi-su.json'
✗ Error in finalizeData: [Errno 2] No such file or directory: 'Resource/VNExpress_CATE.json'

✓ Pipeline completed
